# Análise Exploratória — Acidentes de Trânsito no SUS

**Objetivo:** Validar o caminho escolhido para o pipeline analítico usando amostra de dados.

**Referência CID-10:** V01 a V89 (Acidentes de Transporte Terrestre)

**Fontes:** SIM (óbitos) e SIA (ambulatorial/custos)

## 1. Configuração e Setup

In [ ]:
import sys
from pathlib import Path

# Adiciona raiz do projeto ao path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.logging_config import setup_logging
setup_logging(level="INFO", log_file=True)

import pandas as pd
import duckdb
from config.settings import (
    BRONZE_DIR,
    GOLD_DIR,
    CID_TRANSITO_PREFIX,
    CID_TRANSITO_RANGE,
    MUNICIPIOS_PRIORITARIOS,
)

## 2. Extração de Dados

Tenta download via PySUS. Em caso de falha de rede, usa amostra sintética para validação.

In [ ]:
from data_pipeline.extract import extrair_sim_pysus, gerar_amostra_sim, carregar_sim_bronze

# Tenta download real; fallback para amostra sintética
arquivos = extrair_sim_pysus(ufs=["BA", "SP", "MG"], anos=[2022, 2023])

if not arquivos:
    raise RuntimeError("Nenhum arquivo obtido")

# Carrega primeiro arquivo (ou concatena se múltiplos)
dfs = [carregar_sim_bronze(p) for p in arquivos if p.exists()]
df_sim = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]

print(f"Registros carregados: {len(df_sim):,}")
df_sim.head(10)

## 3. Exploração do Schema

In [ ]:
print("Colunas disponíveis:")
for c in df_sim.columns:
    print(f"  - {c}: {df_sim[c].dtype}")

print("\nAmostra de valores CAUSABAS (causa básica):")
print(df_sim["CAUSABAS"].value_counts().head(20))

print("\nColuna município (ocorrência):")
col_mun = "CODMUNOCOR" if "CODMUNOCOR" in df_sim.columns else "CODMUNRES"
print(df_sim[col_mun].value_counts().head(10))

## 4. Filtro CID V01–V89 (Acidentes de Trânsito)

In [ ]:
from data_pipeline.extract import filtrar_transito

df_transito = filtrar_transito(df_sim)
pct = 100 * len(df_transito) / len(df_sim) if len(df_sim) > 0 else 0
print(f"Registros de trânsito (V01-V89): {len(df_transito):,} ({pct:.1f}% do total)")
df_transito.head()

## 5. Transformação Bronze → Silver → Gold

In [ ]:
from data_pipeline.transform import bronze_to_silver_sim, silver_to_gold_obitos

# Silver: filtrado, tipado, padronizado
df_silver = bronze_to_silver_sim(df_sim)

# Gold: agregado por município e competência
df_gold = silver_to_gold_obitos(df_silver)

print("Gold (óbitos por município/mês):")
df_gold.head(15)

## 6. Validação com DuckDB

In [ ]:
from data_pipeline.load import carregar_gold_duckdb, query_obitos_municipio

conn = duckdb.connect(":memory:")
carregar_gold_duckdb(conn, df_gold)

# Exemplo: óbitos em Vitória da Conquista
vc = query_obitos_municipio(conn, "2933307", ano=2023)
print("Óbitos em Vitória da Conquista (2023):")
print(vc)

# SQL direto
print("\nTop 5 municípios por óbitos:")
conn.execute("""
    SELECT cod_mun_ibge, SUM(obitos) as total_obitos
    FROM gold.v_obitos_transito
    GROUP BY cod_mun_ibge
    ORDER BY total_obitos DESC
    LIMIT 5
""").fetchdf()

## 7. Visualizações (Data Storytelling)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Série temporal - óbitos ao longo do tempo
ts = df_gold.groupby("competencia")["obitos"].sum().reset_index()
ts["competencia"] = pd.to_datetime(ts["competencia"])
axes[0].plot(ts["competencia"], ts["obitos"], marker="o", markersize=4)
axes[0].set_title("Óbitos por Acidentes de Trânsito — Série Temporal")
axes[0].set_xlabel("Competência")
axes[0].set_ylabel("Óbitos")
axes[0].grid(True, alpha=0.3)

# Distribuição por município
mun_totais = df_gold.groupby("cod_mun_ibge")["obitos"].sum().sort_values(ascending=False).head(10)
nomes = [MUNICIPIOS_PRIORITARIOS.get(str(m), str(m)) for m in mun_totais.index]
axes[1].barh(range(len(nomes)), mun_totais.values)
axes[1].set_yticks(range(len(nomes)))
axes[1].set_yticklabels(nomes)
axes[1].set_title("Top 10 Municípios — Total de Óbitos")
axes[1].set_xlabel("Óbitos")

plt.tight_layout()
plt.show()

## 8. Conclusões da Validação

- [x] Schema SIM compatível (CAUSABAS, CODMUNOCOR, DTOBITO)
- [x] Filtro CID V01–V89 aplicável
- [x] Agregação município/competência viável
- [x] DuckDB adequado para consultas OLAP
- [x] Caminho validado para POC